# Modelado y Línea Base (Bank Marketing)

**Grupo:** Grupo 4  
**Archivo del módulo:** `src/modelo.py`  
**Objetivo del notebook:** Implementar los modelos base (trivial y simple) y las Redes Neuronales (MLP v1 y v2), generando la tabla comparativa oficial de la Etapa 3.

In [1]:
import sys
from pathlib import Path

# Añadir la raíz del proyecto al path
raiz_proyecto = Path("..").resolve()
if str(raiz_proyecto) not in sys.path:
    sys.path.insert(0, str(raiz_proyecto))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score

# Importar las funciones de src/modelo.py
from src.modelo import (
    entrenar_modelo_trivial,
    entrenar_baseline_simple,
    construir_mlp_v1,
    construir_mlp_v2,
    construir_preprocesador,
    fijar_semilla,
    entrenar_red
)
from src.evaluacion import calcular_metricas, elegir_umbral

print("¡Librerías y módulos importados correctamente sin errores!")

¡Librerías y módulos importados correctamente sin errores!


### Carga y Partición de Datos
Cargamos el dataset, aplicamos el pipeline de partición y ajustamos el preprocesador (One-Hot de las 5 variables elegidas) **solo con entrenamiento**, para mantener el aislamiento estricto entre entrenamiento y validación.

In [2]:
from src.datos import cargar_y_limpiar_datos, preparar_particiones, VARIABLES_SELECCIONADAS

ruta_csv = Path("../data/bank-full.csv")
X, y = cargar_y_limpiar_datos(ruta_csv)
X_train, X_val, X_test, y_train, y_val, y_test = preparar_particiones(X, y)

# Preprocesador ajustado solo con entrenamiento (mismo que usa entrenar.py)
preprocesador = construir_preprocesador(VARIABLES_SELECCIONADAS)
X_train = preprocesador.fit_transform(X_train)
X_val = preprocesador.transform(X_val)

print(f"Dimensiones de X_train: {X_train.shape}")
print(f"Dimensiones de X_val:   {X_val.shape}")

Dimensiones de X_train: (31638, 33)
Dimensiones de X_val:   (6777, 33)


### Entrenamiento de Modelos Base y Redes Neuronales (Etapa 3)
Entrenaremos cuatro enfoques diferentes:
1. **Modelo Trivial (Dummy):** Predice siempre la clase mayoritaria.
2. **Baseline Simple (Regresión Logística):** Modelo estadístico con clases balanceadas (`class_weight="balanced"`).
3. **Red Neuronal (MLP V1):** Perceptrón multicapa básico con una capa oculta.
4. **Red Neuronal (MLP V2 + Dropout):** Perceptrón más profundo con prevención de sobreajuste.

Las redes usan pesos de clase (equivocarse con un cliente que sí suscribe pesa más), parada temprana (`patience=5`) y semilla fija (42) para que el resultado sea reproducible. Cada modelo se evalúa con su umbral elegido en validación según el costo de los errores (falso negativo = 5, falso positivo = 1).

In [3]:
resultados = []
input_dim = X_train.shape[1]

# --- 1. MODELO TRIVIAL ---
print("Entrenando Modelo Trivial...")
modelo_trivial = entrenar_modelo_trivial(X_train, y_train)
y_pred_trivial = modelo_trivial.predict(X_val)
y_prob_trivial = modelo_trivial.predict_proba(X_val)[:, 1]

resultados.append({
    "Modelo": "1. Trivial (Dummy)",
    "Accuracy": accuracy_score(y_val, y_pred_trivial),
    "Precision": precision_score(y_val, y_pred_trivial, zero_division=0),
    "Recall": recall_score(y_val, y_pred_trivial, zero_division=0),
    "F1-Score": f1_score(y_val, y_pred_trivial, zero_division=0),
    "ROC-AUC": roc_auc_score(y_val, y_prob_trivial),
    "Umbral": None,
    "Costo/cliente": calcular_metricas(y_val, y_pred_trivial, y_prob_trivial)["costo_por_cliente"]
})

# --- 2. MODELO SIMPLE (Regresión Logística) ---
print("Entrenando Baseline Simple...")
modelo_simple = entrenar_baseline_simple(X_train, y_train)
y_prob_simple = modelo_simple.predict_proba(X_val)[:, 1]
umbral_simple = elegir_umbral(y_val, y_prob_simple)
y_pred_simple = (y_prob_simple >= umbral_simple).astype(int)

resultados.append({
    "Modelo": "2. Baseline Simple (LogReg)",
    "Accuracy": accuracy_score(y_val, y_pred_simple),
    "Precision": precision_score(y_val, y_pred_simple, zero_division=0),
    "Recall": recall_score(y_val, y_pred_simple, zero_division=0),
    "F1-Score": f1_score(y_val, y_pred_simple, zero_division=0),
    "ROC-AUC": roc_auc_score(y_val, y_prob_simple),
    "Umbral": umbral_simple,
    "Costo/cliente": calcular_metricas(y_val, y_pred_simple, y_prob_simple)["costo_por_cliente"]
})

# --- 3. MLP V1 ---
print("Entrenando Red Neuronal - MLP Versión 1...")
fijar_semilla(42)
mlp_v1 = construir_mlp_v1(input_dim)
entrenar_red(mlp_v1, X_train, y_train, X_val, y_val)

y_prob_v1 = mlp_v1.predict(X_val, verbose=0).ravel()
umbral_v1 = elegir_umbral(y_val, y_prob_v1)
y_pred_v1 = (y_prob_v1 >= umbral_v1).astype(int)

resultados.append({
    "Modelo": "3. Red Neuronal (MLP V1)",
    "Accuracy": accuracy_score(y_val, y_pred_v1),
    "Precision": precision_score(y_val, y_pred_v1, zero_division=0),
    "Recall": recall_score(y_val, y_pred_v1, zero_division=0),
    "F1-Score": f1_score(y_val, y_pred_v1, zero_division=0),
    "ROC-AUC": roc_auc_score(y_val, y_prob_v1),
    "Umbral": umbral_v1,
    "Costo/cliente": calcular_metricas(y_val, y_pred_v1, y_prob_v1)["costo_por_cliente"]
})

# --- 4. MLP V2 (Con Dropout) ---
print("Entrenando Red Neuronal - MLP Versión 2 (Dropout)...")
fijar_semilla(42)
mlp_v2 = construir_mlp_v2(input_dim)
entrenar_red(mlp_v2, X_train, y_train, X_val, y_val)

y_prob_v2 = mlp_v2.predict(X_val, verbose=0).ravel()
umbral_v2 = elegir_umbral(y_val, y_prob_v2)
y_pred_v2 = (y_prob_v2 >= umbral_v2).astype(int)

resultados.append({
    "Modelo": "4. Red Neuronal (MLP V2 + Dropout)",
    "Accuracy": accuracy_score(y_val, y_pred_v2),
    "Precision": precision_score(y_val, y_pred_v2, zero_division=0),
    "Recall": recall_score(y_val, y_pred_v2, zero_division=0),
    "F1-Score": f1_score(y_val, y_pred_v2, zero_division=0),
    "ROC-AUC": roc_auc_score(y_val, y_prob_v2),
    "Umbral": umbral_v2,
    "Costo/cliente": calcular_metricas(y_val, y_pred_v2, y_prob_v2)["costo_por_cliente"]
})

print("¡Todos los modelos han sido entrenados con éxito!")

Entrenando Modelo Trivial...
Entrenando Baseline Simple...


Entrenando Red Neuronal - MLP Versión 1...


Entrenando Red Neuronal - MLP Versión 2 (Dropout)...


¡Todos los modelos han sido entrenados con éxito!


### Tabla Comparativa de Rendimiento
Generamos la tabla oficial con las métricas clave de evaluación en el conjunto de validación (`Accuracy`, `Precision`, `Recall`, `F1-Score`, `ROC-AUC` y el costo por cliente), cada modelo con su umbral.

In [4]:
df_comparacion = pd.DataFrame(resultados).set_index("Modelo")

print("="*65)
print(" TABLA COMPARATIVA DE RENDIMIENTO DE MODELOS (ETAPA 3) ")
print("="*65)
display(df_comparacion.round(4))

 TABLA COMPARATIVA DE RENDIMIENTO DE MODELOS (ETAPA 3) 


,Accuracy,Precision,Recall,F1-Score,ROC-AUC,Umbral,Costo/cliente
Modelo,,,,,,,
1. Trivial (Dummy),0.8830,0.0000,0.0000,0.0000,0.5000,NaN,0.5851
2. Baseline Simple (LogReg),0.8524,0.3962,0.4981,0.4413,0.7574,0.57,0.3825
3. Red Neuronal (MLP V1),0.8579,0.4138,0.5145,0.4587,0.7623,0.62,0.3693
4. Red Neuronal (MLP V2 + Dropout),0.8541,0.4047,0.5246,0.4569,0.7630,0.58,0.3685


### Discusión de Resultados

* El modelo trivial alcanza una exactitud cercana al 88% sin detectar a ningún cliente que suscribe (Recall = 0): la exactitud no sirve para comparar en este dataset desbalanceado.
* La regresión logística y las dos redes reducen el costo por cliente frente al trivial.
* Las redes superan a la regresión logística, pero por un margen pequeño, y la MLP V2 queda prácticamente igual que la MLP V1. Con solo 5 variables categóricas, una red más profunda no aporta una mejora clara; la regresión logística sigue siendo una alternativa razonable y más fácil de interpretar.
* La versión que se promueve se decide en `entrenar.py` con el criterio de menor costo en validación (ver el notebook 04).